## **Evulations model**

In [13]:
import gc
import torch

# del base_model
# del tuned_model
# del temp_model
# del tokenizer

gc.collect()


2044

In [1]:
import os
import math

import torch
from torch.utils.data import DataLoader

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig,
    default_data_collator
)

from peft import PeftModel

In [2]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
adapter_path = "./tinylama-math-lora-tuned-adapter"


bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# base_model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     quantization_config=bnb_config,
#     device_map="auto",
#     trust_remote_code=True
# ).eval()

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)


# temp_model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     quantization_config=bnb_config,
#     device_map="auto",
#     trust_remote_code=True
# )

# tuned_model = PeftModel.from_pretrained(temp_model, adapter_path)
# tuned_model = tuned_model.merge_and_unload().eval()

In [3]:
def tokenize(batch):
    texts = [
        f"### Instruction:\n{instruction}\n### Response:\n{out}"
        for instruction, out in zip(batch["question"], batch["answer"])
    ]

    tokens = tokenizer(
        texts,
        padding="max_length",
        max_length=256,
        truncation=True,
        return_tensors="pt"
    )

    tokens["labels"] = tokens["input_ids"].clone()

    return tokens


In [4]:
eval_ds = load_dataset("openai/gsm8k", "main", split="train[:200]")
eval_ds = eval_ds.map(tokenize, batched=True, remove_columns=["question", "answer"])
eval_ds = eval_ds.with_format("torch")

In [5]:
eval_loader = DataLoader(
    eval_ds,
    batch_size=1,
    pin_memory=False
)

## **Perplexity Calcutions**
- lower perplexity is batter performence

In [6]:
def compute_perplexity(model):
    losses = []

    for batch in eval_loader:
        input_ids = batch["input_ids"].to("cuda")
        labels = batch["labels"].to("cuda")

        # Mask instruction part
        labels[input_ids == tokenizer.pad_token_id] = -100

        outputs = model(input_ids=input_ids, labels=labels)
        losses.append(outputs.loss.item())

    return math.exp(sum(losses) / len(losses))




In [7]:
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
).eval()

print(f'Base Model Perplexity: {compute_perplexity(base_model):.2f}')

del base_model
torch.cuda.empty_cache()


Base Model Perplexity: 6.14


In [10]:
temp_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

tuned_model = PeftModel.from_pretrained(temp_model, adapter_path)
# tuned_model = tuned_model.merge_and_unload().eval()
tuned_model = tuned_model.eval()

print(f'Tuned Model Perplexity: {compute_perplexity(tuned_model):.2f}')

# del tuned_model
# torch.cuda.empty_cache()


Tuned Model Perplexity: 1.09


In [11]:
prompt = "### Instruction:\nSolve 2x + 3 = 7\n### Response:\n"
print(tokenizer.decode(
    tuned_model.generate(
        **tokenizer(prompt, return_tensors="pt").to("cuda"),
        max_new_tokens=64
    )[0]
))


<s> ### Instruction:
Solve 2x + 3 = 7
### Response:
2x + 3 = 7(2+3) = <<2*3=6>>6
6x = 6(2-3) = <<6-6=0>>0
2x = 5(2+3) = <<2*2+3=7>>7
